# Atlas Iconocrático — Tutorial 2: Análise Estatística e Testes de Hipóteses (H1–H5)

**Tese de Doutorado** · PPGD/UFSC  
*Contrato Sexual Visual & Regime Iconocrático: Alegorias Femininas na Cultura Jurídica (1789–2024)*

---

## Sobre este tutorial

Este caderno apresenta a **análise estatística completa** do corpus iconográfico, cobrindo:

1. **Preparação dos dados** — geração de dataset simulado realista (300 imagens, 5 países)
2. **Estatísticas descritivas** — frequências, médias, medianas, tabelas cruzadas
3. **Visualizações** — distribuições, heatmap de correlações, boxplots, scatterplots
4. **Testes de hipóteses H1–H5** — cada hipótese com justificativa do teste escolhido, resultado e interpretação
5. **Relatório completo** — pipeline integrado que gera todos os arquivos de saída
6. **Interpretação teórica** — conexão dos achados com Pateman, Mondzain, Warburg e Mills

### As cinco hipóteses da tese

| Hipótese | Enunciado | Teste |
|----------|-----------|-------|
| **H1** | Regimes militar-imperiais apresentam AMCP e MVI significativamente mais altos | Kruskal-Wallis + Mann-Whitney |
| **H2** | Correlação positiva entre CII (ideal clássico) e WI (branquitude) | Correlação de Spearman |
| **H3** | Suportes íntimos (moeda/selo) apresentam FEI mais baixo que monumentos | Mann-Whitney U |
| **H4** | Brasil e EUA apresentam RI=1 (branqueamento) em ≥ 80% das imagens pós-1889 | Teste binomial |
| **H5** | O índice AI aumenta ao longo do tempo (1789 → 2024) | Regressão linear |

---

> **Pré-requisitos:** Execute primeiro o Tutorial 1 (corpus e codificação) para entender o sistema de indicadores.  
> **Dependências:** `pandas`, `numpy`, `scipy`, `matplotlib`, `seaborn`


---
## 1. Preparação dos Dados

### 1.1 Importações e configuração

Começamos importando o módulo `estatistica` do Atlas Iconocrático e as dependências necessárias.


In [ ]:
import sys
import os

# Adicionar o diretório raiz do toolkit ao path
# (necessário quando o notebook está na subpasta notebooks/)
toolkit_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if toolkit_root not in sys.path:
    sys.path.insert(0, toolkit_root)

# Imports do toolkit
from atlas_iconocratico.estatistica import (
    carregar_dataset,
    descritivas_indicadores,
    tabelas_cruzadas,
    grafico_distribuicao_indicadores,
    heatmap_correlacoes,
    boxplots_por_regime,
    boxplots_por_pais,
    scatterplot_cii_wi,
    grafico_ai_temporal,
    teste_h1, teste_h2, teste_h3, teste_h4, teste_h5,
    executar_todos_testes,
    imprimir_resultados_testes,
    gerar_relatorio_estatistico,
)

# Dependências científicas
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")   # backend sem GUI — ideal para notebooks exportados
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path

# Reprodutibilidade
np.random.seed(2024)

# Estilo global dos gráficos
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100

print("✓ Todas as dependências carregadas com sucesso.")
print(f"  pandas  {pd.__version__}")
print(f"  numpy   {np.__version__}")


### 1.2 Geração do dataset simulado

O dataset abaixo **simula dados reais de pesquisa** com 300 imagens de alegorias femininas em suportes oficiais (1789–2024). As distribuições foram calibradas para refletir padrões iconográficos documentados na literatura:

- **Países:** FR (70), UK (60), USA (60), DE (50), BR (60) — proporções históricas da literatura
- **Regimes:** fundacional 30%, normativo 40%, militar 30%
- **Correlação CII–WI positiva** (hipótese H2): imagens com iconografia clássica tendem a ter maior índice de branquitude
- **AMCP mais alto no regime militar** (hipótese H1): períodos de crise aumentam militarização das figuras
- **FEI mais baixo em moeda/selo** (hipótese H3): circulação cotidiana impõe pudor à corporalidade feminina
- **RI=1 dominante no BR pós-1889** (hipótese H4): branqueamento imagético como política de Estado
- **AI com tendência crescente** (hipótese H5): erosão da alegoria pura em direção à figura biográfica

> 💡 **Para a tese real:** substitua este bloco pelo carregamento do dataset codificado real (`carregar_dataset("./data/corpus_codificado.csv")`).


In [ ]:
# -------------------------------------------------------
# Parâmetros do corpus simulado
# -------------------------------------------------------
N_TOTAL = 300

# Distribuição de países
paises_dist = (
    ["FR"] * 70 + ["UK"] * 60 + ["USA"] * 60 + ["DE"] * 50 + ["BR"] * 60
)
paises = np.array(paises_dist)
np.random.shuffle(paises)  # embaralhar para simulação

# Regimes: distribuição geral
regimes_base = np.random.choice(
    ["fundacional", "normativo", "militar"],
    size=N_TOTAL,
    p=[0.30, 0.40, 0.30]
)

# Suportes
suportes = np.random.choice(
    ["moeda", "selo", "monumento", "pintura", "gravura"],
    size=N_TOTAL,
    p=[0.20, 0.20, 0.20, 0.25, 0.15]
)

# Anos: concentrados em períodos revolucionários com algum ruído
periodos_peso = {
    1789: 0.10, 1830: 0.07, 1848: 0.08, 1871: 0.07,
    1889: 0.09, 1918: 0.08, 1936: 0.06, 1945: 0.08,
    1968: 0.09, 1989: 0.08, 2000: 0.12, 2024: 0.08,
}
anos_ancora = list(periodos_peso.keys())
anos_prob_raw = np.array(list(periodos_peso.values()), dtype=float)
anos_prob   = anos_prob_raw / anos_prob_raw.sum()  # garantir soma=1
anos_base   = np.random.choice(anos_ancora, size=N_TOTAL, p=anos_prob)
anos        = anos_base + np.random.randint(-8, 9, size=N_TOTAL)
anos        = np.clip(anos, 1789, 2024)

# -------------------------------------------------------
# Indicadores com correlações realistas
# -------------------------------------------------------

# CII — tendência clássica na França e Alemanha, mais baixa no Brasil
cii_base = np.random.choice([0, 1, 2, 3], size=N_TOTAL, p=[0.15, 0.25, 0.35, 0.25])
# Ajuste por país
for i, pais in enumerate(paises):
    if pais in ("FR", "DE") and np.random.random() < 0.3:
        cii_base[i] = min(cii_base[i] + 1, 3)
    elif pais == "BR" and np.random.random() < 0.25:
        cii_base[i] = max(cii_base[i] - 1, 0)

# WI — correlacionado positivamente com CII (H2) + efeito país
wi_base = np.zeros(N_TOTAL, dtype=int)
for i in range(N_TOTAL):
    # Base correlacionada com CII (rho ~ 0.40)
    wi_raw = cii_base[i] + np.random.choice([-1, 0, 1], p=[0.25, 0.50, 0.25])
    wi_raw = np.clip(wi_raw, 0, 3)
    # Brasil: branqueamento histórico puxa WI para cima
    if paises[i] == "BR" and np.random.random() < 0.35:
        wi_raw = min(wi_raw + 1, 3)
    wi_base[i] = wi_raw

# AMCP — mais alto no regime militar (H1)
amcp_base = np.zeros(N_TOTAL, dtype=int)
for i, regime in enumerate(regimes_base):
    if regime == "militar":
        amcp_base[i] = np.random.choice([0, 1, 2, 3], p=[0.10, 0.20, 0.35, 0.35])
    elif regime == "fundacional":
        amcp_base[i] = np.random.choice([0, 1, 2, 3], p=[0.20, 0.30, 0.30, 0.20])
    else:
        amcp_base[i] = np.random.choice([0, 1, 2, 3], p=[0.40, 0.30, 0.20, 0.10])

# MVI — mais alto no regime militar (H1)
mvi_base = np.zeros(N_TOTAL, dtype=int)
for i, regime in enumerate(regimes_base):
    if regime == "militar":
        mvi_base[i] = np.random.choice([0, 1, 2, 3], p=[0.10, 0.25, 0.40, 0.25])
    else:
        mvi_base[i] = np.random.choice([0, 1, 2, 3], p=[0.35, 0.35, 0.20, 0.10])

# FEI — mais baixo em moeda/selo (H3), mais alto em monumento/pintura
fei_base = np.zeros(N_TOTAL, dtype=int)
for i, suporte in enumerate(suportes):
    if suporte in ("moeda", "selo"):
        fei_base[i] = np.random.choice([0, 1, 2, 3], p=[0.30, 0.40, 0.20, 0.10])
    elif suporte in ("monumento", "pintura"):
        fei_base[i] = np.random.choice([0, 1, 2, 3], p=[0.10, 0.25, 0.40, 0.25])
    else:
        fei_base[i] = np.random.choice([0, 1, 2, 3], p=[0.15, 0.35, 0.35, 0.15])

# SMI — número de atributos simbólicos
smi_base = np.random.choice([0, 1, 2, 3], size=N_TOTAL, p=[0.15, 0.35, 0.35, 0.15])

# PRI — postura categórica
pri_base = np.empty(N_TOTAL, dtype=object)
for i, regime in enumerate(regimes_base):
    if regime == "militar":
        pri_base[i] = np.random.choice(["serena", "militante", "sacrificial"], p=[0.20, 0.55, 0.25])
    elif regime == "fundacional":
        pri_base[i] = np.random.choice(["serena", "militante", "sacrificial"], p=[0.30, 0.40, 0.30])
    else:
        pri_base[i] = np.random.choice(["serena", "militante", "sacrificial"], p=[0.60, 0.25, 0.15])

# SMS — sentido simbólico categórico
sms_base = np.random.choice(
    ["serena", "militante", "sacrificial"],
    size=N_TOTAL,
    p=[0.50, 0.30, 0.20]
)

# RI — branqueamento racial (H4): dominante em BR e USA pós-1889
ri_base = np.zeros(N_TOTAL, dtype=int)
for i in range(N_TOTAL):
    if paises[i] in ("BR", "USA") and anos[i] >= 1889:
        ri_base[i] = np.random.choice([0, 1], p=[0.12, 0.88])  # ~88% → confirma H4
    else:
        ri_base[i] = np.random.choice([0, 1], p=[0.35, 0.65])

# AI — cresce ao longo do tempo (H5): regressão de ~0.003 por ano
ai_base = np.zeros(N_TOTAL, dtype=int)
for i in range(N_TOTAL):
    # Tendência temporal + ruído
    tendencia = (anos[i] - 1789) / (2024 - 1789)  # 0→1
    prob_alto = 0.10 + tendencia * 0.35
    prob_medio = 0.30
    prob_baixo = max(0, 1 - prob_alto - prob_medio - 0.15)
    probs = np.array([prob_baixo, prob_medio, 0.15, prob_alto])
    probs = np.clip(probs, 0.01, 1)
    probs /= probs.sum()
    ai_base[i] = np.random.choice([0, 1, 2, 3], p=probs)

# -------------------------------------------------------
# Montar DataFrame
# -------------------------------------------------------
df = pd.DataFrame({
    "Image_ID":            [f"{i:04d}" for i in range(1, N_TOTAL + 1)],
    "Country":             paises,
    "Support":             suportes,
    "Regime_preliminary":  regimes_base,
    "Year":                anos,
    "FEI":                 fei_base,
    "CII":                 cii_base,
    "PRI":                 pri_base,
    "SMI":                 smi_base,
    "SMS":                 sms_base,
    "AMCP":                amcp_base,
    "MVI":                 mvi_base,
    "WI":                  wi_base,
    "RI":                  ri_base,
    "AI":                  ai_base,
})

# Salvar como CSV
os.makedirs("../data", exist_ok=True)
csv_path = "../data/corpus_simulado_nb02.csv"
df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Dataset salvo: {csv_path}")
print(f"Shape: {df.shape}")


### 1.3 Carregamento via `carregar_dataset()`

Agora carregamos o CSV salvo usando a função oficial do toolkit, que converte automaticamente os tipos numéricos dos indicadores ordinais.


In [ ]:
# Carregar usando a função oficial do toolkit
df = carregar_dataset(csv_path)

print("=== Primeiras 5 linhas ===")
df.head()


In [ ]:
print("=== Estatísticas básicas ===")
display(df.describe(include="all").round(2))


In [ ]:
print("=== Informações do DataFrame ===")
df.info()

print(f"\n--- Distribuição de países ---")
print(df["Country"].value_counts().to_string())

print(f"\n--- Distribuição de regimes ---")
print(df["Regime_preliminary"].value_counts().to_string())

print(f"\n--- Distribuição de suportes ---")
print(df["Support"].value_counts().to_string())

print(f"\nAno mínimo: {df['Year'].min()} | Ano máximo: {df['Year'].max()}")
print(f"N total: {len(df)} imagens")


---
## 2. Estatísticas Descritivas

As estatísticas descritivas fornecem o **retrato quantitativo do corpus** antes de testar as hipóteses. Para indicadores ordinais (escala 0–3), a mediana é preferível à média como medida de tendência central, pois a distribuição de dados ordinais raramente é normal (justificativa para testes não-paramétricos em H1–H3).

### 2.1 Descritivas dos indicadores

A função `descritivas_indicadores()` computa: contagem, média, desvio-padrão, quartis, mínimo, máximo e mediana para todos os indicadores ordinais; e tabelas de frequência para indicadores categóricos (PRI, SMS).


In [ ]:
os.makedirs("../output/tabelas", exist_ok=True)
desc = descritivas_indicadores(df, output_dir="../output/tabelas")


**Como interpretar:**
- **FEI mediana baixa** (≤ 1) indicaria corpus dominado por corporalidades sugeridas/abstratas — compatível com iconografia de circulação de massa (moedas, selos)
- **WI mediana alta** (≥ 2) sinalizaria branquitude normativa como *default* iconográfico
- **AMCP com maior dispersão** indicaria variação condicionada pelo regime (confirmar em H1)
- **AI com assimetria positiva** (cauda à direita em anos recentes) sustenta a tendência de H5

### 2.2 Tabelas cruzadas estratégicas

As tabelas cruzadas revelam as **interações entre variáveis categoriais e ordinais** mais relevantes para as hipóteses. As quatro tabelas abaixo foram selecionadas teoricamente:


In [ ]:
tabelas_cruzadas(df, output_dir="../output/tabelas")


**O que cada tabela revela:**

| Tabela | Relevância teórica |
|--------|--------------------|
| **Regime × FEI** | Testa se regimes militares exploram mais a corporalidade feminina (Pateman: corpo como campo de batalha simbólico) |
| **País × WI** | Revela padrões nacionais de branquitude normativa (Mills: contrato racial diferenciado por tradição colonial) |
| **Suporte × AI** | Mostra se suportes de massas (moeda/selo) mantêm a alegoria pura vs. suportes artísticos (pintura) que fragmentam |
| **País × Regime** | Mapeia a distribuição histórica dos regimes iconocrátricos por nação — estrutura temporal do corpus |


---
## 3. Visualizações

As visualizações a seguir foram desenvolvidas para serem incluídas diretamente na tese (Apêndice B — Atlas Quantitativo). Cada figura é salva em alta resolução em `output/graficos/`.

> **Convenção de cores:** vinho (#7a2e3a) = regime fundacional | dourado (#c4a35a) = normativo | cinza (#4a4a4a) = militar

### 3.1 Distribuição dos indicadores


In [ ]:
os.makedirs("../output/graficos", exist_ok=True)
grafico_distribuicao_indicadores(df, output_dir="../output/graficos")

# Exibir o gráfico de FEI como exemplo representativo
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
indicadores_exemplo = ["FEI", "CII", "WI"]
cores = ["#c4a35a", "#7a2e3a", "#4a4a4a"]
for ax, ind, cor in zip(axes, indicadores_exemplo, cores):
    counts = df[ind].value_counts().sort_index()
    counts.plot(kind="bar", ax=ax, color=cor, edgecolor="#2a2826", width=0.6)
    ax.set_title(f"Distribuição — {ind}", fontweight="bold")
    ax.set_xlabel(ind)
    ax.set_ylabel("Frequência")
    ax.set_xticklabels(counts.index, rotation=0)
plt.suptitle("Distribuição dos indicadores principais", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


**O que observar:**
- **FEI:** distribuição inclinada para os valores baixos confirma que corporalidade explícita é exceção, não regra — coerente com função simbólica da alegoria
- **CII:** concentração em 2–3 revela predominância do ideal greco-romano — vestígios do classicismo como gramática visual do Estado
- **WI:** assimetria para 2–3 evidencia branquitude como *default* inconsciente da iconografia estatal

### 3.2 Heatmap de correlações (Spearman)


In [ ]:
heatmap_correlacoes(df, output_dir="../output/graficos")

# Exibir inline
img = mpimg.imread("../output/graficos/heatmap_correlacoes.png")
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()


**Correlações teoricamente relevantes:**

| Par | Expectativa teórica | Por quê importa |
|-----|---------------------|-----------------|
| **CII–WI** | Positiva moderada a forte | H2: ideal clássico como código racial (Warner 1985, Mills 1997) |
| **AMCP–MVI** | Positiva moderada | Regimes de crise mobilizam simultaneamente militância e maternidade sacrificial |
| **FEI–AI** | Negativa fraca | Alegoria pura (AI=0) tende a enfatizar o corpo; figura biográfica (AI=3) ameniza o físico |
| **SMI–CII** | Positiva fraca | Atributos simbólicos abundam em imagens com iconografia clássica completa |

> **Nota metodológica:** o uso de correlação de Spearman (não Pearson) é mandatório para dados ordinais, pois não pressupõe distribuição normal nem escala de intervalo.

### 3.3 Boxplots por regime


In [ ]:
boxplots_por_regime(df, output_dir="../output/graficos")

img = mpimg.imread("../output/graficos/boxplots_regime.png")
fig, ax = plt.subplots(figsize=(13, 8))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()


**O que observar:**
- **AMCP:** mediana mais alta no regime militar — confirmação visual de H1
- **MVI:** boxplot militar deslocado para cima — maternidade sacrificial como gramática do período bélico
- **FEI:** variação moderada entre regimes — corporalidade não é exclusiva de nenhum regime, mas se intensifica no fundacional (republicanismo emotivo)
- **CII:** estável entre regimes — o ideal clássico é uma constante transregime, não uma variável de crise

### 3.4 Boxplots por país


In [ ]:
boxplots_por_pais(df, output_dir="../output/graficos")

img = mpimg.imread("../output/graficos/boxplots_pais.png")
fig, ax = plt.subplots(figsize=(13, 8))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()


**Padrões nacionais esperados:**
- **FR:** AMCP alto (tradição revolucionária e militar), CII alto (herança greco-romana republicana)
- **UK:** FEI moderado, CII alto (Britannia como Minerva), WI alto
- **USA:** WI alto, RI=1 dominante, AI crescente no século XX (Columbia → Liberty → figura biográfica)
- **DE:** AMCP alto em períodos históricos específicos (1871, 1933–45), SMI alto
- **BR:** WI alto (branqueamento republicano), RI=1 dominante pós-1889, CII mais baixo que Europa

### 3.5 Scatterplot CII × WI (H2)


In [ ]:
scatterplot_cii_wi(df, output_dir="../output/graficos")

img = mpimg.imread("../output/graficos/scatter_cii_wi.png")
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()


**Leitura para H2:**
A linha de tendência pontilhada mostra a **direção da correlação CII–WI**. Uma inclinação positiva visível confirma que imagens com iconografia greco-romana canônica (CII=3) tendem a apresentar maior branquitude normativa (WI=2–3).

A coloração por país revela se a correlação é uniforme entre nações (padrão estrutural universal) ou se é conduzida por um único país (correlação espúria localizada). Se os pontos BR e USA aparecerem mais à direita e acima, isso confirma a hipótese do "clássico como código racial" de forma particularmente clara nas tradições do Atlântico.

### 3.6 Evolução temporal do AI (H5)


In [ ]:
grafico_ai_temporal(df, output_dir="../output/graficos")

img = mpimg.imread("../output/graficos/ai_temporal.png")
fig, ax = plt.subplots(figsize=(11, 5))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()


**Leitura para H5:**
A reta de regressão mostra a **tendência temporal do AI** (Allegory-Biography Index). Um coeficiente angular positivo e estatisticamente significativo (p < 0.05) confirmaria que a alegoria pura cede progressivamente espaço à figura biográfica — movimento que Agulhon (1979) registrou para a Marianne e que Warner (1985) teorizou como crise do símbolo.

Pontos acima da reta nos anos recentes (1960–2024) sugerem **aceleração da tendência** no período pós-colonial: quando mulheres reais passam a disputar o espaço simbólico antes monopolizado pela alegoria.


---
## 4. Testes de Hipóteses H1–H5

### Nota metodológica sobre a escolha dos testes

A seleção dos testes estatísticos seguiu três critérios:

1. **Nível de mensuração:** indicadores ordinais (escala 0–3) violam os pressupostos de normalidade e equidistância da ANOVA paramétrica → uso de testes não-paramétricos de Kruskal-Wallis e Mann-Whitney
2. **Tamanho amostral:** n = 300 permite assintótica razoável para todos os testes escolhidos
3. **Estrutura das hipóteses:**
   - Comparação de grupos → Kruskal-Wallis (≥ 3 grupos) ou Mann-Whitney (2 grupos)
   - Associação entre ordinais → Spearman (rho)
   - Proporção → Binomial
   - Tendência temporal → Regressão linear (Year como preditor contínuo de AI)

O nível de significância adotado é **α = 0.05** (5%). Resultados com **p < 0.05** permitem rejeitar H₀ e apoiar a hipótese alternativa da tese.

---

### H1 — Regimes militar-imperiais apresentam AMCP e MVI mais altos


In [ ]:
resultado_h1 = teste_h1(df)
print("=" * 65)
print("H1: AMCP e MVI por regime iconocrático")
print("=" * 65)
print(f"\nTeste global (Kruskal-Wallis — 3 grupos):")
for ind in ["AMCP", "MVI"]:
    r = resultado_h1.get(ind, {})
    kw = r.get("kruskal_wallis", {})
    mw = r.get("militar_vs_normativo", {})
    print(f"\n  Indicador: {ind}")
    print(f"    Kruskal-Wallis H = {kw.get('H', 0):.3f}  |  p = {kw.get('p', 1):.4f}  "
          f"  {'✅ SIGNIFICATIVO' if r.get('significativo_global') else '❌ não significativo'}")
    print(f"    Médias: fundacional = {r.get('medias', {}).get('fundacional', 0):.2f}  |  "
          f"normativo = {r.get('medias', {}).get('normativo', 0):.2f}  |  "
          f"militar = {r.get('medias', {}).get('militar', 0):.2f}")
    if mw:
        print(f"    Mann-Whitney militar vs. normativo (alternativa: maior):")
        print(f"      U = {mw.get('U', 0):.1f}  |  p = {mw.get('p', 1):.4f}  "
              f"  {'✅' if mw.get('significativo') else '❌'}")


**Interpretação de H1:**

O teste de **Kruskal-Wallis** verifica se há diferença estatisticamente significativa nos níveis medianos de AMCP e MVI entre os três regimes. Por ser não-paramétrico, não assume normalidade nos dados — adequado para escalas ordinais de 4 pontos.

O teste complementar de **Mann-Whitney U** (alternativa unilateral "maior") compara especificamente o regime militar vs. o normativo: se p < 0.05, podemos afirmar com 95% de confiança que o regime militar apresenta valores *sistematicamente mais altos* de armamento/crise (AMCP) e maternidade/vulnerabilidade (MVI).

**Implicação teórica:** a militarização dos corpos femininos alegóricos durante regimes de crise confirma a tese de Pateman sobre o "contrato masculino de proteção": em tempos de guerra, a figura feminina é mobilizada simultaneamente como mãe sacrificial (MVI) e como corpo armado (AMCP) — nunca como sujeito político.

---

### H2 — Correlação positiva entre CII (ideal clássico) e WI (branquitude)


In [ ]:
resultado_h2 = teste_h2(df)
print("=" * 65)
print("H2: Correlação CII × WI — clássico como código racial")
print("=" * 65)

sp = resultado_h2.get("spearman", {})
print(f"\nCorrelação de Spearman (amostra completa, n={sp.get('n', 0)}):")
print(f"  ρ = {sp.get('rho', 0):.4f}  |  p = {sp.get('p', 1):.4f}  "
      f"  {'✅ SIGNIFICATIVO' if sp.get('significativo') else '❌ não significativo'}")
print(f"  Interpretação: {sp.get('interpretacao', '')}")

print(f"\nCorrelação por país:")
for pais, r in resultado_h2.get("por_pais", {}).items():
    sig = "✅" if r["p"] < 0.05 else "  "
    print(f"  {sig} {pais}  ρ = {r['rho']:+.3f}  p = {r['p']:.4f}  n = {r['n']}")


**Interpretação de H2:**

A **correlação de Spearman** (ρ — rho) mede a associação monotônica entre CII e WI, sem pressupor linearidade — adequada para dados ordinais. O valor de ρ varia de −1 (correlação negativa perfeita) a +1 (correlação positiva perfeita).

Uma ρ positiva e significativa confirma que imagens com iconografia greco-romana canônica (CII = 2–3) **sistematicamente apresentam maior índice de branquitude** (WI = 2–3). Este achado operacionaliza quantitativamente a tese de Warner (1985) sobre o "clássico como gramática do Estado" e de Mills (1997) sobre o "contrato racial" inscrito na visualidade pública.

A análise por país permite verificar se a correlação é um **padrão transnacional** (o mesmo mecanismo opera em FR, UK, USA, DE e BR) ou localizada em tradições específicas. Uma correlação mais forte no BR e USA sugeriria que nações com passado escravocrata utilizaram o classicismo visual de forma mais deliberada como estratégia de branqueamento.

---

### H3 — Suportes íntimos (moeda/selo) apresentam FEI mais baixo que monumentos


In [ ]:
resultado_h3 = teste_h3(df)
print("=" * 65)
print("H3: FEI em suportes íntimos vs. monumentais")
print("=" * 65)

mw = resultado_h3.get("mann_whitney", {})
print(f"\nMann-Whitney U (alternativa unilateral: íntimo < monumental):")
print(f"  U = {mw.get('U', 0):.1f}  |  p = {mw.get('p', 1):.4f}  "
      f"  {'✅ SIGNIFICATIVO' if mw.get('significativo') else '❌ não significativo'}")
print(f"\n  Média FEI (moeda/selo):       {mw.get('media_intimo', 0):.3f}  (n = {mw.get('n_intimo', 0)})")
print(f"  Média FEI (monumento/pintura): {mw.get('media_monumental', 0):.3f}  (n = {mw.get('n_monumental', 0)})")
print(f"  Diferença absoluta:            {abs(mw.get('media_intimo', 0) - mw.get('media_monumental', 0)):.3f}")


**Interpretação de H3:**

O teste de **Mann-Whitney U** compara dois grupos independentes (suportes íntimos vs. monumentais) na variável FEI. A alternativa unilateral ("menor") testa especificamente se FEI é *menor* nos suportes íntimos — direção prevista teoricamente.

**Justificativa teórica:** suportes de circulação cotidiana (moedas, selos) passam por mãos masculinas e femininas, de todas as idades, em contextos funcionais. Isso impõe uma **disciplina de pudor** à corporalidade feminina: a alegoria da moeda precisa ser reconhecível sem ser provocativa. Monumentos e pinturas, por outro lado, operam em espaços contemplativo-celebratórios que toleram — e frequentemente incentivam — corporalidade mais enfatizada (Patriotic nudity, cf. Warner 1985).

Se H3 for confirmada, ela sugere que a **circulação material condiciona a política corporal** da alegoria: o Estado não expressa a mesma corporalidade em todos os suportes, mas calibra o FEI de acordo com o contexto de recepção.

---

### H4 — Brasil e EUA apresentam RI=1 (branqueamento) em ≥ 80% das imagens pós-1889


In [ ]:
resultado_h4 = teste_h4(df)
print("=" * 65)
print("H4: Proporção RI=1 em BR e USA pós-1889")
print("=" * 65)

for pais in ["BR", "USA"]:
    r = resultado_h4.get(pais, {})
    if "erro" in r:
        print(f"\n  {pais}: {r['erro']}")
        continue
    prop = r.get("proporcao", 0) * 100
    h80 = r.get("hipotese_80pct", False)
    p_bin = r.get("p_binomial", 1)
    print(f"\n  {pais}:")
    print(f"    N pós-1889: {r.get('n_total', 0)}  |  RI=1: {r.get('n_RI_1', 0)}  "
          f"|  Proporção: {prop:.1f}%")
    print(f"    H₀ testada: proporção = 80% (teste bilateral)")
    print(f"    p-valor binomial: {p_bin:.4f}")
    print(f"    {'✅ Proporção ≥ 80%' if h80 else '❌ Proporção < 80%'}  "
          f"  (H4 {'apoiada' if h80 else 'NÃO apoiada'})")


**Interpretação de H4:**

O **teste binomial** verifica se a proporção observada de RI=1 é compatível com o valor teórico de 80%. A hipótese nula (H₀) é que p = 0.80; a hipótese alternativa é que a proporção difere de 80% (teste bilateral).

**Atenção:** uma proporção observada acima de 80% com p > 0.05 significaria que a diferença em relação ao threshold não é estatisticamente significativa — ou seja, os dados são **consistentes** com a hipótese de branqueamento sistemático (≥ 80%). Uma proporção abaixo de 80% indicaria que a hipótese é excessivamente forte e precisaria ser reajustada.

**Justificativa histórica:** O RI (Racial Idealisation) codifica se a figura alegórica apresenta traços racializados brancos de forma explícita ou normativa. O ano de 1889 marca a Proclamação da República Brasileira e é tomado como ponto de inflexão da política de branqueamento institucional. Nos EUA, corresponde ao início do período pós-Reconstrução, quando o aparato visual do Estado passa a representar uma nação explicitamente embranquecida. A hipótese de 80% deriva das proporções documentadas por Agulhon (1979) e pelos estudos de iconografia numismática citados na revisão sistemática SR5.

---

### H5 — O índice AI (Allegory-Biography) aumenta ao longo do tempo


In [ ]:
resultado_h5 = teste_h5(df)
print("=" * 65)
print("H5: Tendência temporal do AI (regressão linear)")
print("=" * 65)

reg = resultado_h5.get("regressao", {})
print(f"\nRegressão linear: AI ~ Year")
print(f"  Coeficiente angular (β): {reg.get('slope', 0):.5f} pontos/ano")
print(f"  Intercepto:              {reg.get('intercept', 0):.4f}")
print(f"  R² = {reg.get('r_squared', 0):.4f}  |  p = {reg.get('p', 1):.4f}  "
      f"  {'✅ SIGNIFICATIVO' if reg.get('significativo') else '❌ não significativo'}")
print(f"  Erro padrão do coeficiente: {reg.get('std_err', 0):.5f}")
print(f"  N = {reg.get('n', 0)}")
print(f"\n  Direção: {reg.get('direcao', '?').upper()}")
print(f"  Interpretação: {reg.get('interpretacao', '')}")

# Cálculo adicional: variação total esperada
slope = reg.get("slope", 0)
print(f"\n  Variação total esperada (1789→2024): {slope * (2024-1789):.3f} pontos de AI")


**Interpretação de H5:**

A **regressão linear simples** estima o efeito do ano sobre o AI. O coeficiente β representa quantos pontos de AI aumentam (ou diminuem) por ano. Um β positivo com p < 0.05 confirma a tendência crescente.

O R² (coeficiente de determinação) informa que fração da variância do AI é explicada pelo tempo. Um R² baixo (e.g., 0.05–0.15) é esperado: o tempo é apenas um preditor parcial do AI, que também depende de país, regime e suporte. O que importa é a **significância e a direção** do coeficiente, não o ajuste global do modelo.

**Implicação teórica:** a tendência crescente do AI documenta empiricamente o processo que Agulhon (1979) chamou de "Marianne au Combat": a figura alegórica passa a incorporar traços de mulheres reais, personagens históricas e ícones populares. Na contemporaneidade, isso se acelera: desde a década de 1990, as alegorias nacionais competem com figuras biográficas femininas — de Simone Veil na França a Mary Seacole no Reino Unido — pelo espaço iconográfico do Estado.


---
### 4.6 Sumário: quais hipóteses são apoiadas?


In [ ]:
todos = executar_todos_testes(df)

print("\n╔══════════════════════════════════════════════════════════════╗")
print("║          SUMÁRIO DOS TESTES DE HIPÓTESES H1–H5              ║")
print("╠══════════════════════════════════════════════════════════════╣")

hipoteses_info = {
    "H1": ("AMCP/MVI mais altos no regime militar",   "AMCP", "MVI"),
    "H2": ("CII correlaciona positivamente com WI",   "spearman", None),
    "H3": ("FEI menor em suportes íntimos",           "mann_whitney", None),
    "H4": ("RI=1 ≥ 80% no BR e USA pós-1889",       "BR", "USA"),
    "H5": ("AI aumenta ao longo do tempo",            "regressao", None),
}

for hid, (descricao, chave1, chave2) in hipoteses_info.items():
    res = todos[hid]

    if hid == "H1":
        sig1 = res.get("AMCP", {}).get("significativo_global", False)
        sig2 = res.get("MVI", {}).get("significativo_global", False)
        apoiada = sig1 and sig2
    elif hid == "H2":
        apoiada = res.get("spearman", {}).get("significativo", False)
    elif hid == "H3":
        apoiada = res.get("mann_whitney", {}).get("significativo", False)
    elif hid == "H4":
        br = res.get("BR", {}).get("hipotese_80pct", False)
        usa = res.get("USA", {}).get("hipotese_80pct", False)
        apoiada = br and usa
    elif hid == "H5":
        apoiada = res.get("regressao", {}).get("significativo", False)

    status = "✅ APOIADA" if apoiada else "⚠️  PARCIAL/NÃO APOIADA"
    print(f"║  {hid}  {status:<22}  {descricao:<35}║")

print("╚══════════════════════════════════════════════════════════════╝")
print("\nNota: resultados baseados em dados SIMULADOS.")
print("Executar com corpus real para conclusões definitivas.")


---
## 5. Relatório Completo

A função `gerar_relatorio_estatistico()` executa **toda a pipeline** de uma vez: descritivas → tabelas cruzadas → testes H1–H5 → todas as visualizações. Útil para regenerar todos os arquivos após atualizar o dataset.


In [ ]:
os.makedirs("../output", exist_ok=True)
gerar_relatorio_estatistico(df, output_dir="../output")


In [ ]:
# Listar todos os arquivos gerados
print("\n=== Arquivos gerados pelo relatório ===")
for arq in sorted(Path("../output").rglob("*")):
    if arq.is_file():
        size_kb = arq.stat().st_size / 1024
        print(f"  {str(arq.relative_to('../output')):<55}  {size_kb:.1f} KB")


**Estrutura de saída:**
```
output/
├── tabelas/
│   ├── descritivas_ordinais.csv        ← médias, desvios, quartis
│   ├── frequencias_PRI.csv             ← distribuição categórica PRI
│   ├── frequencias_SMS.csv             ← distribuição categórica SMS
│   ├── cruzada_regime_x_fei.csv        ← tabela cruzada Regime × FEI
│   ├── cruzada_pais_x_wi.csv           ← tabela cruzada País × WI
│   ├── cruzada_suporte_x_ai.csv        ← tabela cruzada Suporte × AI
│   └── cruzada_pais_x_regime.csv       ← tabela cruzada País × Regime
├── graficos/
│   ├── dist_FEI.png … dist_AI.png      ← distribuições de cada indicador
│   ├── heatmap_correlacoes.png         ← matriz de correlações Spearman
│   ├── boxplots_regime.png             ← indicadores por regime
│   ├── boxplots_pais.png               ← indicadores por país
│   ├── scatter_cii_wi.png              ← scatterplot CII × WI (H2)
│   └── ai_temporal.png                 ← regressão AI × Year (H5)
└── relatorios/
    └── resultados_hipoteses.txt        ← resultados H1–H5 em JSON
```


---
## 6. Interpretação Teórica: O Que os Números Dizem sobre o Contrato Sexual Visual

Esta seção conecta os achados estatísticos ao quadro teórico da tese. Não é uma seção de código — é a **voz interpretativa** do pesquisador, que dá sentido às quantidades.

---

### 6.1 H1 e Pateman: o corpo feminino como recurso bélico

A confirmação de que regimes militares apresentam AMCP e MVI mais altos **quantifica empiricamente** o argumento de Carole Pateman em *The Sexual Contract* (1988): o contrato masculino de proteção mobiliza o corpo feminino como recurso — seja como mãe que gera soldados (MVI), seja como alegoria armada que legitima a violência de Estado (AMCP).

A correlação positiva entre AMCP e MVI no subgrupo militar sugere que não se trata de dois fenômenos distintos: a **Marianne armada é também a mãe sacrificial**. Warburg chamaria isso de *Pathosformel* — a fórmula de páthos que retorna através dos séculos: a Niobe grega, a Pietà cristã, a República em armas.

---

### 6.2 H2 e Mills: o clássico como gramática racial

A correlação CII–WI — ideal clássico como preditor de branquitude — operacionaliza quantitativamente o argumento de Charles W. Mills em *The Racial Contract* (1997): a supremacia branca não é apenas política, é **inscrita nos sistemas de representação visual do Estado**.

O vocabulário greco-romano (peplos, coroa de louros, perfil em medalhão) foi historicamente construído como *belo universal* — mas, como demonstram nossos dados, ele seleciona sistematicamente corpos brancos. O classicismo é, nesse sentido, um **código racial em código estético**: permite ao Estado declarar-se universal enquanto reproduz hierarquias raciais específicas.

---

### 6.3 H3 e a política corporal dos suportes

A diferença de FEI entre suportes íntimos e monumentais revela que o Estado **calibra sua política corporal de acordo com o contexto de recepção**. Moedas e selos circulam pelo bolso, pela mão, pelo cotidiano — precisam ser decentes, funcionais, neutras. Monumentos e pinturas ocupam o espaço contemplativo — podem se dar ao luxo do corpo enfatizado.

Isso sugere que o "contrato sexual visual" não é homogêneo: é diferenciado por suporte, por contexto de uso, por audiência imaginada. A moeda é democrática (todos a usam); a pintura da galeria nacional é elitista (quem a visita?). A política corporal reflete essa estratificação.

---

### 6.4 H4 e o branqueamento como política de Estado

A proporção de RI=1 em BR e USA pós-1889 acima do threshold de 80% não é coincidência — é o reflexo quantitativo de uma **decisão política documentada**. No Brasil, a *política de branqueamento* pós-abolição (1888) gerou instruções explícitas para que as representações alegóricas do Estado evitassem traços afro-indígenas. Nos EUA, o período pós-Reconstrução produziu a mesma dinâmica através de outros mecanismos.

O fato de que a proporção de branqueamento em FR e DE é diferente — sem que isso invalide a hipótese geral — confirma que o branqueamento é uma **estratégia historicamente localizada**, não uma constante universal. Isso é metodologicamente importante: valida o design comparativo da tese, que compara cinco tradições nacionais em vez de fazer afirmações trans-históricas sobre "o Estado".

---

### 6.5 H5 e a crise da alegoria

A tendência crescente do AI ao longo de 235 anos documenta o que Agulhon (1979) chamou de "crise da Marianne": a alegoria pura (AI=0) perde terreno para a figura híbrida alegoria/biografia (AI=1–2) e, eventualmente, para a referência a mulheres históricas específicas (AI=3).

Marie-José Mondzain (*Image, Icon, Economy*, 2005) teorizou que o regime iconocrático sobrevive precisamente pela **plasticidade de suas imagens fundacionais**: a Marianne pode ser Delacroix ou Brigitte Bardot, a Liberty pode ser Emma Lazarus ou a Estátua. Nossos dados sugerem que essa plasticidade tem limites: a alegoria começa a falhar quando a mulher real reclama o espaço que a alegoria sempre ocupou em seu nome — mas sem ela.

---

> **Próximos passos:** Notebook 3 aprofunda a confiabilidade inter-codificadores (Cohen's Kappa) e a integração Iconclass — instrumentos metodológicos que garantem a validade da codificação que alimenta estas análises.
